In [1]:
import pandas as pd
import os

In [2]:
df=pd.read_csv(r'C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\final_processed_data.csv')

In [3]:
def get_match_phase(over):
    """
    Categorize over numbers into Powerplay, Middle, and Death.
    """
    try:
        if over <= 6:
            return "Powerplay"
        elif over <= 15:
            return "Middle"
        else:
            return "Death"
    except:
        return None

In [4]:
df['oversActual'] = pd.to_numeric(df['oversActual'], errors='coerce').fillna(0)


In [5]:
df['match_phase'] = df['oversActual'].apply(get_match_phase)

In [6]:
if 'run' not in df.columns:
    if 'totalRuns' in df.columns:
        df['run'] = df['totalRuns']
    else:
        raise KeyError("No run or totalRuns column found!")


In [7]:
df.columns

Index(['Unnamed: 0', 'match_obj_id', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Full Name', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Full Name_bowler', 'Batting Style (s)_bowler', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role', 'match_phase'],
      dtype='object')

In [8]:
df["valid_delivery"] = ~((df["wides"] > 0) | (df["noballs"] > 0))

In [9]:
df['isBoundary']=df['isFour']|df['isSix']

In [10]:
phase_sr = (
    df[df["valid_delivery"]]
    .groupby(["Batsman_Name", "match_phase"])
    .agg(
        total_runs=("run", "sum"),
        balls_faced=("valid_delivery", "count"),
        boundaries=('isBoundary', 'sum'),
        wickets=('isWicket', 'sum')
    )
    .reset_index()
)

In [11]:
phase_sr['boundary_percentage'] = (phase_sr['boundaries'] / phase_sr['balls_faced']) * 100
phase_sr['wicket_percentage'] = (phase_sr['wickets'] / phase_sr['balls_faced']) * 100
phase_sr['dot_ball_percentage'] = 100 - (phase_sr['boundary_percentage'] + phase_sr['wicket_percentage'])

In [12]:
phase_sr["strike_rate"] = (phase_sr["total_runs"] / phase_sr["balls_faced"]) * 100
phase_sr["strike_rate"] = phase_sr["strike_rate"].fillna(0).round(2)

In [13]:
for col in ['strike_rate', 'boundary_percentage', 'dot_ball_percentage']:
    phase_sr[col + '_norm'] = (
        (phase_sr[col] - phase_sr[col].min()) / 
        (phase_sr[col].max() - phase_sr[col].min())
    )

phase_sr['adaptability_index'] = (
    0.5 * phase_sr['strike_rate_norm'] +
    0.3 * phase_sr['boundary_percentage_norm'] +
    0.2 * phase_sr['dot_ball_percentage_norm']
).round(3)

phase_sr['adaptability_label'] = pd.cut(
    phase_sr['adaptability_index'],
    bins=[-0.1, 0.4, 0.7, 1.0],
    labels=['Low', 'Moderate', 'High']
)


In [14]:
role_map = (
        df.groupby("Full Name")["Batsman_Playing_Role"]
        .agg(lambda x: x.mode().iloc[0] if len(x.mode()) else None)
        .to_dict()
    )
phase_sr["Batsman_Playing_Role"] = phase_sr["Batsman_Name"].map(role_map)

In [15]:
phase_sr.head()

,Batsman_Name,match_phase,total_runs,balls_faced,boundaries,wickets,boundary_percentage,wicket_percentage,dot_ball_percentage,strike_rate,strike_rate_norm,boundary_percentage_norm,dot_ball_percentage_norm,adaptability_index,adaptability_label,Batsman_Playing_Role
0,A Balbirnie,Death,4,8,0,3,0.000000,37.500000,62.500000,50.00,0.115386,0.000000,0.625000,0.183,Low,NaN
1,A Balbirnie,Middle,111,94,11,4,11.702128,4.255319,84.042553,118.09,0.272517,0.117021,0.840426,0.339,Low,NaN
2,A Balbirnie,Powerplay,130,120,17,5,14.166667,4.166667,81.666667,108.33,0.249994,0.141667,0.816667,0.331,Low,NaN
3,A Bohara,Death,0,1,0,1,0.000000,100.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.000,Low,NaN
4,A Dutt,Death,16,11,2,0,18.181818,0.000000,81.818182,145.45,0.335656,0.181818,0.818182,0.386,Low,NaN


In [16]:
phase_sr.to_csv(r'C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\phase_sr.csv')

In [17]:
top_batsmen = (
    phase_sr.groupby("Batsman_Name")["strike_rate"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .index
)


In [18]:
top_batsmen

Index(['L Ngidi', 'PM Nevill', 'Asif Ali', 'Mudassar Bukhari', 'Mohammad Sami',
       'MJ McClenaghan', 'Mohammad Saifuddin', 'T Panyangara', 'N Pooran',
       'CR Brathwaite'],
      dtype='object', name='Batsman_Name')

In [19]:
import plotly.express as px

In [20]:
fig = px.bar(
    phase_sr[phase_sr["Batsman_Name"].isin(top_batsmen)],
    x="Batsman_Name",
    y="strike_rate",
    color="match_phase",
    barmode="group",
    title="🏏 Top 10 Batsmen - Phase-wise Strike Rate",
    labels={"strike_rate": "Strike Rate", "Batsman_Name": "Batsman", "match_phase": "Phase"},
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()


In [21]:
pivot = phase_sr.pivot(index="Batsman_Name", columns="match_phase", values="strike_rate").fillna(0)
pivot = pivot.loc[top_batsmen]

fig2 = px.imshow(
    pivot,
    text_auto=".1f",
    color_continuous_scale="RdYlGn",
    title="🔥 Batsman Phase Performance Heatmap (Strike Rate)",
    labels=dict(x="Phase", y="Batsman", color="Strike Rate")
)
fig2.show()